# Anotação assistida no Google Colab

Versão para o Colab do `anotacao/anotar.py`: o [Grounding DINO](https://huggingface.co/IDEA-Research/grounding-dino-tiny) propõe caixas no formato do YOLO para as fotos de uma pasta. As caixas são **propostas** para um revisor humano, não rótulos finais. Detalhes (classes, saída, opções) em [`anotacao/README.md`](https://github.com/vdlwang/ecovision-iterativo/blob/main/anotacao/README.md).

O notebook não copia o código: ele clona o repositório e roda o próprio `anotar.py`, com as classes de `anotacao/classes.yaml`. Rode as células em ordem.

## 1. GPU

Em *Ambiente de execução → Alterar o tipo de ambiente de execução*, escolha uma GPU (a T4 gratuita basta). Sem GPU funciona, só que bem mais devagar.

In [ ]:
!nvidia-smi -L || echo "Sem GPU: a anotação vai rodar na CPU"

## 2. Código

Clona o repositório (ou atualiza, se a célula já rodou). O `torch` e o `transformers` já vêm instalados no Colab.

In [ ]:
RAMO = "feature/anotacao-assistida"  # @param {type:"string"}

import os, sys

DIR_REPO = "/content/ecovision-iterativo"
if os.path.isdir(DIR_REPO):
    !git -C {DIR_REPO} fetch -q --depth 1 origin {RAMO} && git -C {DIR_REPO} checkout -q FETCH_HEAD
else:
    !git clone -q --depth 1 -b {RAMO} https://github.com/vdlwang/ecovision-iterativo.git {DIR_REPO}
!git -C {DIR_REPO} log --oneline -1

DIR_ANOTACAO = f"{DIR_REPO}/anotacao"
if DIR_ANOTACAO not in sys.path:
    sys.path.insert(0, DIR_ANOTACAO)

## 3. Imagens

Pastas no Google Drive (`/content/drive/MyDrive/...`) sobrevivem ao fim da sessão; pastas fora dele (`/content/...`) somem quando o Colab desconecta. Com *enviar_do_computador*, as fotos escolhidas são copiadas para `pasta_imagens`.

In [ ]:
pasta_imagens = "/content/drive/MyDrive/ecovision/fotos"  # @param {type:"string"}
pasta_saida = "/content/drive/MyDrive/ecovision/anotacoes"  # @param {type:"string"}
enviar_do_computador = False  # @param {type:"boolean"}

from pathlib import Path

if any(p.startswith("/content/drive") for p in (pasta_imagens, pasta_saida)):
    from google.colab import drive
    drive.mount("/content/drive")

pasta_imagens, pasta_saida = Path(pasta_imagens), Path(pasta_saida)
pasta_imagens.mkdir(parents=True, exist_ok=True)

if enviar_do_computador:
    from google.colab import files
    for nome, conteudo in files.upload().items():
        (pasta_imagens / nome).write_bytes(conteudo)

from visualizar import EXTENSOES_IMAGEM

imagens = sorted(p for p in pasta_imagens.iterdir()
                 if p.is_file() and p.suffix.lower() in EXTENSOES_IMAGEM)
print(f"{len(imagens)} imagem(ns) em {pasta_imagens}")

## 4. Anotar

As mesmas opções do `anotar.py`. Sem *sobrescrever*, as imagens que já têm `.txt` em `pasta_saida` são puladas, então dá para rodar de novo depois de uma desconexão e continuar de onde parou. Na primeira vez, os pesos do modelo (~700 MB para o `tiny`) são baixados do Hugging Face.

In [ ]:
modelo = "IDEA-Research/grounding-dino-tiny"  # @param ["IDEA-Research/grounding-dino-tiny", "IDEA-Research/grounding-dino-base"]
confianca = 0.35  # @param {type:"slider", min:0.05, max:0.95, step:0.05}
iou = 0.5  # @param {type:"slider", min:0.1, max:0.95, step:0.05}
area_maxima = 0.5  # @param {type:"slider", min:0.05, max:1.0, step:0.05}
previa = True  # @param {type:"boolean"}
sobrescrever = False  # @param {type:"boolean"}
classes = ""  # @param {type:"string"}
# classes vazio: anotacao/classes.yaml do repositório

import anotar

pendentes = imagens if sobrescrever else [
    p for p in imagens if not (pasta_saida / "labels" / f"{p.stem}.txt").exists()]
print(f"{len(imagens) - len(pendentes)} já anotada(s), {len(pendentes)} a anotar")

if pendentes:
    argv = [*map(str, pendentes), "-o", str(pasta_saida), "--modelo", modelo,
            "--confianca", str(confianca), "--iou", str(iou),
            "--area-maxima", str(area_maxima)]
    if classes:
        argv += ["--classes", classes]
    if previa:
        argv.append("--previa")
    if sobrescrever:
        argv.append("--sobrescrever")
    try:
        anotar.main(argv)
    except ValueError as erro:
        print(f"Erro: {erro}")
    except SystemExit:
        pass  # o argparse já imprimiu o motivo

## 5. Conferir

Mostra as prévias (exige *previa* marcado no passo 4). O `visualizar.html` gerado em `pasta_saida` não abre dentro do Colab: baixe a pasta (passo 6) ou abra pelo Google Drive no computador.

In [ ]:
quantas = 6  # @param {type:"integer"}
primeira = 0  # @param {type:"integer"}

from IPython.display import Image, display

previas = sorted((pasta_saida / "previa").glob("*.jpg"))
print(f"{len(previas)} prévia(s)")
for p in previas[primeira:primeira + quantas]:
    print(p.name)
    display(Image(filename=str(p), width=640))

## 6. Baixar

Compacta `pasta_saida` num `.zip` e baixa para o computador. Dentro dele, abra `visualizar.html` para navegar pelas anotações.

In [ ]:
import shutil
from google.colab import files

arquivo = shutil.make_archive(f"/content/{pasta_saida.name}", "zip", pasta_saida)
files.download(arquivo)